In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

OUT_DIR = Path("../data/processed")

oa = pd.read_csv(OUT_DIR / "oa_counts_master_v1.csv", low_memory=False)

In [15]:
def safe_divide(numerator, denominator):
    return np.where(denominator > 0, numerator / denominator, np.nan)


def sum_cols(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")

    return df[cols].sum(axis=1)

In [16]:
features = oa[["OA21CD"]].copy()

In [17]:
features["total_residents"] = oa["ts001_residence_type_total"]
features["household_residents"] = oa["ts001_residence_type_lives_in_a_household"]
features["communal_establishment_residents"] = oa["ts001_residence_type_lives_in_a_communal_establishment"]

features["communal_establishment_pct"] = safe_divide(
    features["communal_establishment_residents"],
    features["total_residents"]
)

In [18]:
features["age_0_14_count"] = sum_cols(oa, [
    "ts007a_age_aged_4_years_and_under",
    "ts007a_age_aged_5_to_9_years",
    "ts007a_age_aged_10_to_14_years",
])

features["age_15_24_count"] = sum_cols(oa, [
    "ts007a_age_aged_15_to_19_years",
    "ts007a_age_aged_20_to_24_years",
])

features["age_25_34_count"] = sum_cols(oa, [
    "ts007a_age_aged_25_to_29_years",
    "ts007a_age_aged_30_to_34_years",
])

features["age_35_49_count"] = sum_cols(oa, [
    "ts007a_age_aged_35_to_39_years",
    "ts007a_age_aged_40_to_44_years",
    "ts007a_age_aged_45_to_49_years",
])

features["age_50_64_count"] = sum_cols(oa, [
    "ts007a_age_aged_50_to_54_years",
    "ts007a_age_aged_55_to_59_years",
    "ts007a_age_aged_60_to_64_years",
])

features["age_65_plus_count"] = sum_cols(oa, [
    "ts007a_age_aged_65_to_69_years",
    "ts007a_age_aged_70_to_74_years",
    "ts007a_age_aged_75_to_79_years",
    "ts007a_age_aged_80_to_84_years",
    "ts007a_age_aged_85_years_and_over",
])

for col in [
    "age_0_14_count",
    "age_15_24_count",
    "age_25_34_count",
    "age_35_49_count",
    "age_50_64_count",
    "age_65_plus_count",
]:
    features[col.replace("_count", "_pct")] = safe_divide(
        features[col],
        features["total_residents"]
    )

In [19]:
features["partnership_total"] = oa["ts002_marital_and_civil_partnership_status_total"]

features["never_married_count"] = oa[
    "ts002_marital_and_civil_partnership_status_never_married_and_never_registered_a_civil_partnership"
]

features["married_or_civil_partnership_count"] = oa[
    "ts002_marital_and_civil_partnership_status_married_or_in_a_registered_civil_partnership"
]

features["separated_count"] = oa[
    "ts002_marital_and_civil_partnership_status_separated_but_still_legally_married_or_still_legally_in_a_civil_partnership"
]

features["divorced_or_dissolved_count"] = oa[
    "ts002_marital_and_civil_partnership_status_divorced_or_civil_partnership_dissolved"
]

features["widowed_count"] = oa[
    "ts002_marital_and_civil_partnership_status_widowed_or_surviving_civil_partnership_partner"
]

for col in [
    "never_married_count",
    "married_or_civil_partnership_count",
    "separated_count",
    "divorced_or_dissolved_count",
    "widowed_count",
]:
    features[col.replace("_count", "_pct")] = safe_divide(
        features[col],
        features["partnership_total"]
    )

In [20]:
features["household_composition_total"] = oa["ts003_household_composition_total"]

features["one_person_household_count"] = oa[
    "ts003_household_composition_one_person_household"
]

features["one_person_66_plus_count"] = oa[
    "ts003_household_composition_one_person_household_aged_66_years_and_over"
]

features["single_family_household_count"] = oa[
    "ts003_household_composition_single_family_household"
]

features["married_couple_family_count"] = oa[
    "ts003_household_composition_single_family_household_married_or_civil_partnership_couple"
]

features["cohabiting_couple_family_count"] = oa[
    "ts003_household_composition_single_family_household_cohabiting_couple_family"
]

features["lone_parent_family_count"] = oa[
    "ts003_household_composition_single_family_household_lone_parent_family"
]

features["other_household_types_count"] = oa[
    "ts003_household_composition_other_household_types"
]

for col in [
    "one_person_household_count",
    "one_person_66_plus_count",
    "single_family_household_count",
    "married_couple_family_count",
    "cohabiting_couple_family_count",
    "lone_parent_family_count",
    "other_household_types_count",
]:
    features[col.replace("_count", "_pct")] = safe_divide(
        features[col],
        features["household_composition_total"]
    )

In [21]:
features["country_of_birth_total"] = oa["ts004_country_of_birth_total"]

features["uk_born_count"] = oa["ts004_country_of_birth_europe_united_kingdom"]

features["eu_born_count"] = oa["ts004_country_of_birth_europe_eu_countries"]

features["non_uk_born_count"] = (
    features["country_of_birth_total"] - features["uk_born_count"]
)

features["non_uk_non_eu_born_count"] = (
    features["non_uk_born_count"] - features["eu_born_count"]
)

for col in [
    "uk_born_count",
    "eu_born_count",
    "non_uk_born_count",
    "non_uk_non_eu_born_count",
]:
    features[col.replace("_count", "_pct")] = safe_divide(
        features[col],
        features["country_of_birth_total"]
    )

In [22]:
features["length_residence_total"] = oa[
    "ts016_length_of_residence_in_the_uk_total_all_usual_residents"
]

features["born_in_uk_count"] = oa[
    "ts016_length_of_residence_in_the_uk_born_in_the_uk"
]

features["resident_10_plus_years_count"] = oa[
    "ts016_length_of_residence_in_the_uk_10_years_or_more"
]

features["resident_5_to_10_years_count"] = oa[
    "ts016_length_of_residence_in_the_uk_5_years_or_more_but_less_than_10_years"
]

features["resident_2_to_5_years_count"] = oa[
    "ts016_length_of_residence_in_the_uk_2_years_or_more_but_less_than_5_years"
]

features["resident_less_2_years_count"] = oa[
    "ts016_length_of_residence_in_the_uk_less_than_2_years"
]

features["resident_less_5_years_count"] = (
    features["resident_2_to_5_years_count"]
    + features["resident_less_2_years_count"]
)

for col in [
    "born_in_uk_count",
    "resident_10_plus_years_count",
    "resident_5_to_10_years_count",
    "resident_2_to_5_years_count",
    "resident_less_2_years_count",
    "resident_less_5_years_count",
]:
    features[col.replace("_count", "_pct")] = safe_divide(
        features[col],
        features["length_residence_total"]
    )

In [23]:
features["ethnic_group_total"] = oa[
    "ts021_ethnic_group_total_all_usual_residents"
]

features["asian_count"] = oa[
    "ts021_ethnic_group_asian_asian_british_or_asian_welsh"
]

features["black_count"] = oa[
    "ts021_ethnic_group_black_black_british_black_welsh_caribbean_or_african"
]

features["mixed_count"] = oa[
    "ts021_ethnic_group_mixed_or_multiple_ethnic_groups"
]

features["white_count"] = oa[
    "ts021_ethnic_group_white"
]

features["white_british_count"] = oa[
    "ts021_ethnic_group_white_english_welsh_scottish_northern_irish_or_british"
]

features["white_other_count"] = oa[
    "ts021_ethnic_group_white_other_white"
]

features["other_ethnic_group_count"] = oa[
    "ts021_ethnic_group_other_ethnic_group"
]

features["non_white_count"] = (
    features["ethnic_group_total"] - features["white_count"]
)

for col in [
    "asian_count",
    "black_count",
    "mixed_count",
    "white_count",
    "white_british_count",
    "white_other_count",
    "other_ethnic_group_count",
    "non_white_count",
]:
    features[col.replace("_count", "_pct")] = safe_divide(
        features[col],
        features["ethnic_group_total"]
    )

In [24]:
features["accommodation_total"] = oa[
    "ts044_accommodation_type_total_all_households"
]

features["detached_count"] = oa[
    "ts044_accommodation_type_detached"
]

features["semi_detached_count"] = oa[
    "ts044_accommodation_type_semi_detached"
]

features["terraced_count"] = oa[
    "ts044_accommodation_type_terraced"
]

features["purpose_built_flat_count"] = oa[
    "ts044_accommodation_type_in_a_purpose_built_block_of_flats_or_tenement"
]

features["converted_flat_count"] = sum_cols(oa, [
    "ts044_accommodation_type_part_of_a_converted_or_shared_house_including_bedsits",
    "ts044_accommodation_type_part_of_another_converted_building_for_example_former_school_church_or_warehouse",
])

features["commercial_building_flat_count"] = oa[
    "ts044_accommodation_type_in_a_commercial_building_for_example_in_an_office_building_hotel_or_over_a_shop"
]

features["caravan_mobile_temp_count"] = oa[
    "ts044_accommodation_type_a_caravan_or_other_mobile_or_temporary_structure"
]

features["house_type_count"] = (
    features["detached_count"]
    + features["semi_detached_count"]
    + features["terraced_count"]
)

features["flat_type_count"] = (
    features["purpose_built_flat_count"]
    + features["converted_flat_count"]
    + features["commercial_building_flat_count"]
)

for col in [
    "detached_count",
    "semi_detached_count",
    "terraced_count",
    "purpose_built_flat_count",
    "converted_flat_count",
    "commercial_building_flat_count",
    "caravan_mobile_temp_count",
    "house_type_count",
    "flat_type_count",
]:
    features[col.replace("_count", "_pct")] = safe_divide(
        features[col],
        features["accommodation_total"]
    )

C:\Users\keena\AppData\Local\Temp\ipykernel_24144\3252383089.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[col.replace("_count", "_pct")] = safe_divide(


In [25]:
features["tenure_total"] = oa[
    "ts054_tenure_of_household_total_all_households"
]

features["owned_count"] = oa[
    "ts054_tenure_of_household_owned"
]

features["owns_outright_count"] = oa[
    "ts054_tenure_of_household_owned_owns_outright"
]

features["owns_mortgage_count"] = oa[
    "ts054_tenure_of_household_owned_owns_with_a_mortgage_or_loan"
]

features["shared_ownership_count"] = oa[
    "ts054_tenure_of_household_shared_ownership"
]

features["social_rented_count"] = oa[
    "ts054_tenure_of_household_social_rented"
]

features["private_rented_count"] = oa[
    "ts054_tenure_of_household_private_rented"
]

features["lives_rent_free_count"] = oa[
    "ts054_tenure_of_household_lives_rent_free"
]

for col in [
    "owned_count",
    "owns_outright_count",
    "owns_mortgage_count",
    "shared_ownership_count",
    "social_rented_count",
    "private_rented_count",
    "lives_rent_free_count",
]:
    features[col.replace("_count", "_pct")] = safe_divide(
        features[col],
        features["tenure_total"]
    )

C:\Users\keena\AppData\Local\Temp\ipykernel_24144\1101615535.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["tenure_total"] = oa[
C:\Users\keena\AppData\Local\Temp\ipykernel_24144\1101615535.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["owned_count"] = oa[
C:\Users\keena\AppData\Local\Temp\ipykernel_24144\1101615535.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at o

In [26]:
features["occupation_total"] = oa[
    "ts063_occupation_current_total_all_usual_residents_aged_16_years_and_over_in_employment_the_week_before_the_census"
]

features["managerial_professional_count"] = sum_cols(oa, [
    "ts063_occupation_current_1_managers_directors_and_senior_officials",
    "ts063_occupation_current_2_professional_occupations",
    "ts063_occupation_current_3_associate_professional_and_technical_occupations",
])

features["administrative_secretarial_count"] = oa[
    "ts063_occupation_current_4_administrative_and_secretarial_occupations"
]

features["skilled_traditional_count"] = sum_cols(oa, [
    "ts063_occupation_current_5_skilled_trades_occupations",
    "ts063_occupation_current_8_process_plant_and_machine_operatives",
])

features["routine_service_elementary_count"] = sum_cols(oa, [
    "ts063_occupation_current_6_caring_leisure_and_other_service_occupations",
    "ts063_occupation_current_7_sales_and_customer_service_occupations",
    "ts063_occupation_current_9_elementary_occupations",
])

for col in [
    "managerial_professional_count",
    "administrative_secretarial_count",
    "skilled_traditional_count",
    "routine_service_elementary_count",
]:
    features[col.replace("_count", "_pct")] = safe_divide(
        features[col],
        features["occupation_total"]
    )

C:\Users\keena\AppData\Local\Temp\ipykernel_24144\1263914250.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["occupation_total"] = oa[
C:\Users\keena\AppData\Local\Temp\ipykernel_24144\1263914250.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["managerial_professional_count"] = sum_cols(oa, [
C:\Users\keena\AppData\Local\Temp\ipykernel_24144\1263914250.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance. 

In [27]:
features["economic_activity_total"] = oa[
    "ts066_economic_activity_status_total_all_usual_residents_aged_16_years_and_over"
]

features["employed_count"] = (
    oa["ts066_economic_activity_status_economically_active_excluding_full_time_students_in_employment"]
    + oa["ts066_economic_activity_status_economically_active_and_a_full_time_student_in_employment"]
)

features["unemployed_count"] = (
    oa["ts066_economic_activity_status_economically_active_excluding_full_time_students_unemployed"]
    + oa["ts066_economic_activity_status_economically_active_and_a_full_time_student_unemployed"]
)

features["full_time_student_count"] = (
    oa["ts066_economic_activity_status_economically_active_and_a_full_time_student"]
    + oa["ts066_economic_activity_status_economically_inactive_student"]
)

features["economically_inactive_count"] = oa[
    "ts066_economic_activity_status_economically_inactive"
]

features["retired_count"] = oa[
    "ts066_economic_activity_status_economically_inactive_retired"
]

features["looking_after_home_family_count"] = oa[
    "ts066_economic_activity_status_economically_inactive_looking_after_home_or_family"
]

features["long_term_sick_disabled_count"] = oa[
    "ts066_economic_activity_status_economically_inactive_long_term_sick_or_disabled"
]

for col in [
    "employed_count",
    "unemployed_count",
    "full_time_student_count",
    "economically_inactive_count",
    "retired_count",
    "looking_after_home_family_count",
    "long_term_sick_disabled_count",
]:
    features[col.replace("_count", "_pct")] = safe_divide(
        features[col],
        features["economic_activity_total"]
    )

C:\Users\keena\AppData\Local\Temp\ipykernel_24144\3317308484.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["economic_activity_total"] = oa[
C:\Users\keena\AppData\Local\Temp\ipykernel_24144\3317308484.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["employed_count"] = (
C:\Users\keena\AppData\Local\Temp\ipykernel_24144\3317308484.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all

In [28]:
features["qualification_total"] = oa[
    "ts067_highest_level_of_qualification_total_all_usual_residents_aged_16_years_and_over"
]

features["no_qualifications_count"] = oa[
    "ts067_highest_level_of_qualification_no_qualifications"
]

features["level_1_2_count"] = (
    oa["ts067_highest_level_of_qualification_level_1_and_entry_level_qualifications"]
    + oa["ts067_highest_level_of_qualification_level_2_qualifications"]
)

features["apprenticeship_count"] = oa[
    "ts067_highest_level_of_qualification_apprenticeship"
]

features["level_3_count"] = oa[
    "ts067_highest_level_of_qualification_level_3_qualifications"
]

features["level_4_plus_count"] = oa[
    "ts067_highest_level_of_qualification_level_4_qualifications_and_above"
]

features["other_qualifications_count"] = oa[
    "ts067_highest_level_of_qualification_other_qualifications"
]

for col in [
    "no_qualifications_count",
    "level_1_2_count",
    "apprenticeship_count",
    "level_3_count",
    "level_4_plus_count",
    "other_qualifications_count",
]:
    features[col.replace("_count", "_pct")] = safe_divide(
        features[col],
        features["qualification_total"]
    )

C:\Users\keena\AppData\Local\Temp\ipykernel_24144\3261059801.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["qualification_total"] = oa[
C:\Users\keena\AppData\Local\Temp\ipykernel_24144\3261059801.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["no_qualifications_count"] = oa[
C:\Users\keena\AppData\Local\Temp\ipykernel_24144\3261059801.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joini

In [29]:
cluster_features = [
    # Age
    "age_15_24_pct",
    "age_25_34_pct",
    "age_50_64_pct",
    "age_65_plus_pct",

    # Rootedness / churn
    "uk_born_pct",
    "non_uk_born_pct",
    "resident_10_plus_years_pct",
    "resident_less_5_years_pct",

    # Ethnic / migration context
    "white_british_pct",
    "white_other_pct",
    "non_white_pct",

    # Housing / built form
    "owned_pct",
    "owns_outright_pct",
    "social_rented_pct",
    "private_rented_pct",
    "house_type_pct",
    "flat_type_pct",

    # Class / work
    "managerial_professional_pct",
    "skilled_traditional_pct",
    "routine_service_elementary_pct",

    # Economic position
    "employed_pct",
    "unemployed_pct",
    "full_time_student_pct",
    "retired_pct",
    "long_term_sick_disabled_pct",

    # Education
    "no_qualifications_pct",
    "level_1_2_pct",
    "apprenticeship_pct",
    "level_4_plus_pct",

    # Household / family structure
    "one_person_household_pct",
    "married_couple_family_pct",
    "lone_parent_family_pct",
]

In [30]:
oa_features = features[["OA21CD", "total_residents"] + cluster_features].copy()

In [31]:
missing = oa_features[cluster_features].isna().mean().sort_values(ascending=False)
print(missing.head(20))

age_15_24_pct                     0.0
age_25_34_pct                     0.0
age_50_64_pct                     0.0
age_65_plus_pct                   0.0
uk_born_pct                       0.0
non_uk_born_pct                   0.0
resident_10_plus_years_pct        0.0
resident_less_5_years_pct         0.0
white_british_pct                 0.0
white_other_pct                   0.0
non_white_pct                     0.0
owned_pct                         0.0
owns_outright_pct                 0.0
social_rented_pct                 0.0
private_rented_pct                0.0
house_type_pct                    0.0
flat_type_pct                     0.0
managerial_professional_pct       0.0
skilled_traditional_pct           0.0
routine_service_elementary_pct    0.0
dtype: float64


In [32]:
for col in cluster_features:
    oa_features[col] = oa_features[col].fillna(oa_features[col].median())

In [33]:
for col in cluster_features:
    lower = oa_features[col].quantile(0.01)
    upper = oa_features[col].quantile(0.99)
    oa_features[col] = oa_features[col].clip(lower, upper)

In [34]:
features.to_csv(OUT_DIR / "oa_derived_features_full_v1.csv", index=False)
oa_features.to_csv(OUT_DIR / "oa_cluster_features_v1.csv", index=False)